# Supporting Objects
Create views for:
* rolling calendar of past 12 
* most recent name of each job
* most recent name of each pipeline

In [0]:
create or replace view silver_dev.edm.dim_calendar_rolling_13_months
  (   date_id, date_, year_id, month_of_year
  ,   months_ago comment 'The number of months in the past for the given date (current month = 0)'
  )
    comment 'A rolling calendar of the current and prior 12 months, based on the current timestamp'
    as
    select      cal.date_id, cal.date_, cal.year_id, cal.month_of_year
            ,   dense_rank() over (order by month_id desc) - 1 months_ago
    from        silver_dev.edm.dim_calendar cal
    where      (cal.year_id = year(current_timestamp) and cal.month_of_year <= month(current_timestamp)
        or      cal.year_id = year(current_timestamp) - 1 and cal.month_of_year >= month(current_timestamp))


In [0]:
create or replace view metadata_dev.databricks.jobs_current
    comment 'Current definition of each job in the `system.lakeflow.jobs` table'
    as
    select    j.job_id, j.name, j.change_time, j.description, j.workspace_id
    from      system.lakeflow.jobs j
    join    ( select job_id, max(change_time) change_time from system.lakeflow.jobs group by job_id ) mr
        on    mr.job_id = j.job_id and mr.change_time = j.change_time


In [0]:
create or replace view metadata_dev.databricks.pipelines_current
    comment 'Current definition of each pipeline in the `system.lakeflow.pipelines` table'
    as
    select    pl.pipeline_id, pl.pipeline_type, pl.name, pl.change_time, pl.workspace_id
    from      system.lakeflow.pipelines pl
    join    ( select pipeline_id, max(change_time) change_time from system.lakeflow.pipelines group by pipeline_id ) mr
        on    mr.pipeline_id = pl.pipeline_id and mr.change_time = pl.change_time


In [0]:
create or replace view metadata_dev.databricks.warehouses_current
    comment 'Current definition of each warehouse in the `system.compute.warehouses` table'
    as
    select    wh.warehouse_id, wh.warehouse_type, wh.warehouse_name, wh.change_time, wh.workspace_id, wh.warehouse_size, wh.warehouse_channel
    from      system.compute.warehouses wh
    join    ( select warehouse_id, max(change_time) change_time from system.compute.warehouses group by warehouse_id ) mr
        on    mr.warehouse_id = wh.warehouse_id and mr.change_time = wh.change_time


In [0]:
create or replace view metadata_dev.databricks.clusters_current
    comment 'Current definition of each cluster in the `system.compute.clusters` table'
    as
    select    c.cluster_id, c.cluster_name, c.cluster_source, c.dbr_version, c.driver_node_type, c.worker_node_type
          ,   c.enable_elastic_disk, c.min_autoscale_workers, c.max_autoscale_workers
    from      system.compute.clusters c
    join    ( select cluster_id, max(change_time) change_time from system.compute.clusters group by cluster_id ) mr
        on    mr.cluster_id = c.cluster_id and mr.change_time = c.change_time


In [0]:
select 'clusters' obj, count(*) num_rows from system.compute.clusters
union
select 'clusters_current', count(*) from metadata_dev.databricks.clusters_current
union
select 'warehouses' obj, count(*) num_rows from system.compute.warehouses
union
select 'warehouses_current', count(*) from metadata_dev.databricks.warehouses_current
union
select 'jobs' obj, count(*) from system.lakeflow.jobs
union
select 'jobs_current', count(*) from metadata_dev.databricks.jobs_current
union
select 'pipelines' obj, count(*) from system.lakeflow.pipelines
union
select 'pipelines_current', count(*) from metadata_dev.databricks.pipelines_current
order by obj